# Vergleich der Reward-Metrik-Gewichtung (0.5/0.5 vs. 0.7/0.3 vs. 1.0/0.0)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook analysiert und vergleicht das Encoder-Decoder-Modell (`facebook/mbart-large-50`) unter verschiedenen DPO-Gewichtungsschemata:
- **SFT Baseline**
- **DPO (0.5 Style / 0.5 Semantik)**: Ausgewogene Zielgewichtung
- **DPO (0.7 Style / 0.3 Semantik)**: Verstärkter Fokus auf syntaktische und lexikalische Vereinfachung
- **DPO (1.0 Style / 0.0 Semantik)**: Reine Simplicity-Maximierung ohne explizite Semantik-Strafe

### Forschungsfrage:
Wie verändern sich Simplicity ($R_{\text{style}}$), Semantik-Erhalt ($R_{\text{sem, AS}}$), Referenzähnlichkeit ($Sim_{\text{ref}}$) und Satzstrukturen bei steigender Gewichtung der Simplicity-Metrik?

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)
print(f"Arbeitsverzeichnis: {os.getcwd()}")

SUMMARY_PATH = "results/evaluation/metric_weights_comparison_summary.csv"
DETAILS_PATH = "results/evaluation/metric_weights_comparison_details.csv"
PLOT_PATH = "results/plots/metric_weights_tradeoff_curve.png"

## 1. Ergebnisse der quantitativen Zusammenfassung

In [ ]:
if os.path.exists(SUMMARY_PATH):
    df_summary = pd.read_csv(SUMMARY_PATH)
    display(df_summary)
else:
    print(f"Datei '{SUMMARY_PATH}' noch nicht gefunden. Bitte führe zuerst das SLURM-Evaluationsskript aus:")
    print("sbatch scripts/sbatch/experiments/metric_weights/3_run_full_evaluation.sh")

## 2. Trade-Off Visualisierung: Simplicity vs. Semantik-Erhalt

In [ ]:
if os.path.exists(SUMMARY_PATH):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(df_summary["r_style_mean"], df_summary["r_sem_as_mean"], marker="o", markersize=10, linewidth=2, color="#1f77b4")
    for _, row in df_summary.iterrows():
        ax.annotate(
            row["model_name"],
            (row["r_style_mean"], row["r_sem_as_mean"]),
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
            fontweight="bold",
            fontsize=10,
        )
    ax.set_xlabel("Ø Simplicity Score ($R_{style}$)", fontsize=12)
    ax.set_ylabel("Ø Semantik-Erhalt zur Quelle ($R_{sem, AS}$)", fontsize=12)
    ax.set_title("Simplicity vs. Semantik Trade-Off (Pareto-Frontier)", fontsize=14, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

## 3. Qualitative Stichproben & Beispielsätze

In [ ]:
if os.path.exists(DETAILS_PATH):
    df_details = pd.read_csv(DETAILS_PATH)
    # Zeige die ersten 3 Sätze im Vergleich
    sample_as = df_details["as_text"].unique()[:3]
    for i, as_s in enumerate(sample_as, 1):
        sub = df_details[df_details["as_text"] == as_s]
        print(f"\n{'='*80}\nBeispiel {i}:\nAS-Quelle: {as_s}\nLS-Referenz: {sub['ls_ref_text'].iloc[0]}\n{'-'*80}")
        for _, r in sub.iterrows():
            print(f"[{r['model_name']}] (Simp: {r['r_style']:.3f}, Sem: {r['r_sem_as']:.3f}):\n  -> {r['generated_text']}")